<a href="https://colab.research.google.com/github/YANHONGLU/SP500-TOTAL-RETURN-DRAWDOWN-OPTIMIZATION/blob/main/S%26P_500_Total_Return_Drawdown_Optimization_An_11_Day_Stress_Window_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Project Setup and Data Source


This section imports the required Python libraries and prepares the S&P 500 Total Return Index data source, which includes both price appreciation and reinvested dividends.

In [1]:
!pip install yfinance -q

import yfinance as yf
import pandas as pd

## 2. Downloading S&P 500 Total Return Data

This function downloads historical S&P 500 Total Return Index data and prepares the closing-price series for analysis.

In [2]:
# Download S&P 500
def download_total_return_index():
    data = yf.download(
        "^SP500TR",
        start="2004-12-30",
        end="2025-01-02",
        auto_adjust=False,
        progress=False
    )
    close = data["Close"]
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
    close.index = pd.to_datetime(close.index)
    return close.dropna().sort_index()

## 3. Handling Non-Trading Dates

This function identifies the closest available trading date when the requested date falls on a weekend or market holiday.

In [3]:
# Find the price of the most recent trading day before a given date.
def last_close_on_or_before(close, date):
    eligible = close.loc[close.index <= date]
    actual_date = eligible.index[-1]
    price = float(eligible.iloc[-1])
    return actual_date, price

In [4]:
def first_close_on_or_after(close,date):
    eligible=close.loc[close.index>=date]
    actual_date=eligible.index[0]
    price=float(eligible.iloc[0])
    return actual_date,price

## 4. Identifying the Worst 11-Day Stress Window

This function scans every possible 11-calendar-day window and identifies the period with the lowest compounded market return.

In [5]:
#3. Find out which 11 consecutive days to delete to make the most money.
def find_worst_11_day_skip(daily_returns,start_date,end_date):
  worst_factor=float('inf')
  worst_start=None
  worst_end=None

  last_start=end_date-pd.Timedelta(days=10)

  for window_start in pd.date_range(start_date,last_start,freq='D'):
    window_end=window_start+pd.Timedelta(days=10)

    in_window=(daily_returns.index>=window_start)&(daily_returns.index<=window_end)
    window_returns=daily_returns.loc[in_window]
    window_factor=(1+window_returns).prod()

    if worst_factor>window_factor:
      worst_factor=window_factor
      worst_start=window_start
      worst_end=window_end
  return worst_start,worst_end,worst_factor

## 5. Portfolio Return Calculation and Comparison

This section calculates the normal buy-and-hold portfolio value and compares it with the counterfactual result after removing the worst 11-day market window.

In [6]:
# Calculate
initial_money = 100

start_date = pd.Timestamp("2005-01-01")
end_date = pd.Timestamp("2025-01-01")

# Download the index
close = download_total_return_index()

# Find the actual trading dates and prices
actual_start, start_price = first_close_on_or_after(
    close,
    start_date
)

actual_end, end_price = last_close_on_or_before(
    close,
    end_date
)

# Keep only the investment period
holding_close = close.loc[actual_start:actual_end]

# Calculate daily returns
daily_returns = holding_close.pct_change().dropna()

# Calculate the normal final value
normal_final_value = initial_money * (end_price / start_price)

# Find the worst 11-calendar-day window
worst_start, worst_end, worst_factor = find_worst_11_day_skip(
    daily_returns,
    start_date,
    end_date
)

# Calculate the final value after removing the worst 11 days
optimized_final_value = normal_final_value / worst_factor


In [7]:
print(f"Normal final value: ${normal_final_value:.2f}")
print(f"Worst 11-day window: {worst_start.date()} to {worst_end.date()}")
print(f"Optimized final value: ${optimized_final_value:.2f}")
# Initial investment: $100.00
# Normal final value: $723.37
# Worst 11-day window: 2008-09-29 to 2008-10-09
# Optimized final value: $963.38

Normal final value: $723.37
Worst 11-day window: 2008-09-29 to 2008-10-09
Optimized final value: $963.38
